### This notebook is used render map location on folium

In [1]:
import requests
import folium
import json
import os
import requests
import google.auth
import google.auth.transport.requests
from google.oauth2 import id_token
from IPython.display import display
try:
    import h3
    H3_AVAILABLE = True
except ImportError:
    H3_AVAILABLE = False
    print("Warning: h3 library not found. Hexagons will not be drawn. You can install it with pip install h3 --break-system-packages")

In [2]:
def fetch_and_save_state_notebook(thread_id="default", target_url=None):
    import subprocess
    import json
    import requests
    
    # 1. Config
    base_url = "https://insight-agent-backend-v4-374740535943.asia-south1.run.app/"
    url = target_url or f"{base_url}/api/state"
    params = {"thread_id": thread_id}
    
    headers = {
        "Content-Type": "application/json"
    }

    try:
        result = subprocess.run(['gcloud', 'auth', 'print-identity-token'], capture_output=True, text=True, check=True)
        auth_token = (result.stdout.strip())
        headers["Authorization"] = f"Bearer {auth_token}"
    except Exception as e:
        print(e)

    try:
        print(f"Requesting state from: {url}")
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()

        # 4. Save to file
        filename = f"{thread_id}.json"
        with open(filename, 'w') as f:
            json.dump(data, f, indent=4)
            
        print(f"✅ Success! Saved to {filename}")
        return data

    except Exception as e:
        print(f"❌ Error during request: {e}")
        
        filename = f"{thread_id}.json"
        import os
        if os.path.exists(filename):
            print(f"⚠️ Proceeding with existing local data from {filename} as fallback.")
        else:
            print(f"❌ CRITICAL No local data {filename} found to fallback on.")


In [3]:
def query_copilotkit_agent(prompt_content, threadId, runId):
    import subprocess
    import json
    import requests

    url = "https://insight-agent-backend-v4-374740535943.asia-south1.run.app/agents/copilotkit"
    
    auth_token = None
    try:
        result = subprocess.run(['gcloud', 'auth', 'print-identity-token'], capture_output=True, text=True, check=True)
        auth_token = result.stdout.strip()
    except Exception as e:
        print(f"Error getting auth token: {e}")
        return

    headers = {
        "Authorization": f"Bearer {auth_token}",
        "Content-Type": "application/json"
    }

    payload = {
        "threadId": threadId,
        "runId": runId,
        "messages": [
          {
            "id": "msg_user_0",
            "role": "user",
            "content": prompt_content
          }
        ],
        "state": {},
        "tools": [],
        "context": [],
        "forwardedProps": {}
    }

    try:
        print(f"Sending request to {url}...")
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        print("✅ Success! Response received.")
        return response.json()
    except Exception as e:
        if 'response' in locals() and response is not None:
             print(f"Response status: {response.status_code}")
             print(f"Response text: {response.text}")


In [ ]:
test_queries = [
"Please perform competitor analysis for area Borivali",
  "Feasibility check of Dadar location of Mumbai for opening of new Reliance Digital Store.",
  "Please do catchment analysis for area within drive time of 30 minutes from Andheri station location."
]

In [5]:
from datetime import datetime

# Get current local date and time
now = datetime.now()

# Format as dd_mm_yy_hh_mm
formatted_time = now.strftime("%d_%m_%y_%H_%M")

print(formatted_time)

07_04_26_14_56


In [6]:
threadId = formatted_time
runId = formatted_time
for query in test_queries:
  response = query_copilotkit_agent(query,threadId,runId)
  print("Query: ",query)
  print("Response: ",response)

Sending request to https://insight-agent-backend-v4-374740535943.asia-south1.run.app/agents/copilotkit...
✅ Success! Response received.
Response status: 200
Response text: data: {"type":"RUN_STARTED","threadId":"07_04_26_14_56","runId":"07_04_26_14_56"}

data: {"type":"TOOL_CALL_START","toolCallId":"adk-392a787f-a365-4d1c-8ba0-23685dc3c9e4","toolCallName":"transfer_to_agent"}

data: {"type":"TOOL_CALL_ARGS","toolCallId":"adk-392a787f-a365-4d1c-8ba0-23685dc3c9e4","delta":"{\"agent_name\":\"catchment_analyzer_agent\"}"}

data: {"type":"TOOL_CALL_END","toolCallId":"adk-392a787f-a365-4d1c-8ba0-23685dc3c9e4"}

data: {"type":"TOOL_CALL_RESULT","messageId":"9295717d-e113-4f31-9e6c-2fac42f5c334","toolCallId":"adk-392a787f-a365-4d1c-8ba0-23685dc3c9e4","content":"{\"result\": null}"}

data: {"type":"TOOL_CALL_START","toolCallId":"adk-be7c7e55-2ade-4c9c-b370-897f48cc04aa","toolCallName":"identify_coordinates"}

data: {"type":"TOOL_CALL_ARGS","toolCallId":"adk-be7c7e55-2ade-4c9c-b370-897f48cc04aa"

In [1]:
# fetch_and_save_state_notebook(thread_id="default", target_url="http://localhost:8000/api/state")
fetch_and_save_state_notebook(formatted_time)

NameError: name 'fetch_and_save_state_notebook' is not defined

In [ ]:
def display_map_from_api(url_or_data):
    # 1. Fetch data from your local API
    try:
        if isinstance(url_or_data, str):
            response = requests.get(url_or_data)
            response.raise_for_status() # Check for errors
            data = response.json()
        else:
            data = url_or_data
            
        # Parse the relevant data
        value_data = data.get('_value', data)
        catchment_analysis = value_data.get('catchment_analysis', {})
        competitor_analysis = value_data.get('competitor_analysis', {})
        
        main_analysis = catchment_analysis if catchment_analysis else competitor_analysis
        
        marker_point = main_analysis.get('catchment_marker_point')
        
        # fallback for old data format
        if not marker_point:
            lat = data.get('lat', 19.1190)
            lng = data.get('lng', 72.8468)
            map_text = data.get('message', 'No text provided')
            has_marker = True
            radius_m = None
        else:
            if marker_point and len(marker_point) == 2:
                lat, lng = marker_point[0], marker_point[1]
                map_text = "Analysis Center"
                has_marker = True
            else:
                lat, lng = 0, 0
                map_text = "No mapping data"
                has_marker = False
                
            # Additional details for popup
            insights = main_analysis.get('insights', [])
            if insights:
                map_text += "<br><br><b>Insights:</b><br>" + "<br>".join([str(i) for i in insights])
                
            dataset_insights = main_analysis.get('dataset_insights', {})
            if dataset_insights:
                ds_keys = [k for k in dataset_insights.keys() if k not in ('hex_dataset_ids', 'marker_dataset_ids')]
                if ds_keys:
                    map_text += f"<br><br><b>Datasets:</b> {', '.join(ds_keys)}"
                    
            radius_val = main_analysis.get('catchment_analysis_radius')
            if radius_val is not None:
                try:
                    radius_m = float(radius_val)
                except ValueError:
                    radius_m = None
            else:
                radius_m = None
                
        # 2. Create a folium map centered at the coordinates
        m = folium.Map(location=[lat, lng], zoom_start=14)

        # Keep track of all drawn points to fit bounds automatically
        bounds_points = [[lat, lng]] if lat and lng else []

        # 3. Add a marker with the text response in a popup
        if has_marker:
            folium.Marker(
                [lat, lng],
                popup=folium.Popup(map_text, max_width=400),
                tooltip="Center Point",
                icon=folium.Icon(color="red", icon="info-sign")
            ).add_to(m)
            
            if radius_m is not None and radius_m > 0:
                folium.Circle(
                    radius=radius_m,
                    location=[lat, lng],
                    color="red",
                    fill=True,
                    fill_opacity=0.2
                ).add_to(m)

        # Add items from dataset insights (Processing hexagons and markers)
        h3_drawn = 0
        markers_drawn = 0
        
        # Define a palette of colors for different datasets
        dataset_palette = [
            {'color': 'blue', 'fill_color': 'cyan', 'icon_color': 'blue'},
            {'color': 'green', 'fill_color': 'lightgreen', 'icon_color': 'green'},
            {'color': 'purple', 'fill_color': 'violet', 'icon_color': 'purple'},
            {'color': 'orange', 'fill_color': 'gold', 'icon_color': 'orange'},
            {'color': 'darkred', 'fill_color': 'salmon', 'icon_color': 'darkred'}
        ]
        
        dataset_insights = main_analysis.get('dataset_insights', {})
        ds_idx = 0
        
        for dataset_id, dataset_info in dataset_insights.items():
            if dataset_id == 'hex_dataset_ids' or dataset_id == 'marker_dataset_ids':
                continue
                
            # Pick a color scheme for this dataset
            scheme = dataset_palette[ds_idx % len(dataset_palette)]
            ds_idx += 1
            
            if isinstance(dataset_info, dict):
                dataset_details = dataset_info.get('dataset_details', dataset_id)
                
                # Check for hex codes
                hex_codes = dataset_info.get('hex_codes', [])
                if hex_codes and not H3_AVAILABLE:
                    print(f"Found {len(hex_codes)} hex codes for {dataset_id}, but h3 library is not loaded.")
                elif hex_codes:
                    for hex_data in hex_codes:
                        hex_id = hex_data.get('hex_id')
                        details = hex_data.get('details', '')
                        tag = hex_data.get('tag', '')
                        resolution = hex_data.get('res', hex_data.get('resolution', 9))
                        
                        if not hex_id:
                            continue
                            
                        # If resolving competitor tags from dataset_insights
                        if tag == 'Competitor_Only':
                            color = 'red'
                            fill_color = 'salmon'
                        elif tag == 'Reliance_Only':
                            color = 'blue'
                            fill_color = 'lightblue'
                        elif tag == 'Both_Presence':
                            color = 'purple'
                            fill_color = 'violet'
                        else:
                            color = scheme['color']
                            fill_color = scheme['fill_color']

                        try:
                            try:
                                hex_boundary = h3.cell_to_boundary(hex_id)
                            except AttributeError:
                                hex_boundary = h3.h3_to_geo_boundary(hex_id)
                                
                            popup_content = f"<b>{dataset_details}</b><br>{details}<br>Resolution: {resolution}<br>Tag: {tag}"
                            folium.Polygon(
                                locations=hex_boundary,
                                color=color,
                                weight=2,
                                fill=True,
                                fill_color=fill_color,
                                fill_opacity=0.3,
                                popup=folium.Popup(popup_content, max_width=300),
                                tooltip=details if details else f"Hex: {hex_id}"
                            ).add_to(m)
                            bounds_points.extend(hex_boundary)
                            h3_drawn += 1
                        except Exception as e:
                            pass

                # Check for markers
                markers = dataset_info.get('markers', [])
                for marker_data in markers:
                    m_lat = marker_data.get('lat')
                    m_long = marker_data.get('long', marker_data.get('lon'))
                    if m_lat is not None and m_long is not None:
                        details = marker_data.get('details', '')
                        place_id = marker_data.get('place_id', '')
                        tag = marker_data.get('tag', '')
                        
                        icon_color = scheme['icon_color']
                        icon_prefix = 'glyphicon'
                        icon_name = 'info-sign'
                        
                        if 'reliance_store' in str(tag).lower() or tag == 'Reliance_Only':
                            icon_color = 'blue'
                        elif 'competitor' in str(tag).lower() or tag == 'Competitor_Only':
                            icon_color = 'red'
                            icon_prefix = 'fa'
                            icon_name = 'shopping-cart'

                        popup_content = f"<b>{dataset_details}</b><br>{details}<br>Place ID: {place_id}<br>Tag: {tag}"
                        folium.Marker(
                            [float(m_lat), float(m_long)],
                            popup=folium.Popup(popup_content, max_width=300),
                            tooltip=details if details else f"Marker: {tag}",
                            icon=folium.Icon(color=icon_color, icon=icon_name, prefix=icon_prefix)
                        ).add_to(m)
                        bounds_points.append([float(m_lat), float(m_long)])
                        markers_drawn += 1

        # Process places_insights for markers
        places_insights = main_analysis.get('places_insights', {})
            
        markers = places_insights.get('markers', [])
        for marker_data in markers:
            m_lat = marker_data.get('lat')
            m_long = marker_data.get('long', marker_data.get('lon'))
            if m_lat is not None and m_long is not None:
                details = marker_data.get('details', '')
                place_id = marker_data.get('place_id', '')
                tag = marker_data.get('tag', '')
                
                popup_content = f"<b>Places Insight</b><br>{details}<br>Place ID: {place_id}<br>Tag: {tag}"
                folium.Marker(
                    [float(m_lat), float(m_long)],
                    popup=folium.Popup(popup_content, max_width=300),
                    tooltip=details if details else f"Place: {tag}",
                    icon=folium.Icon(color="darkblue", icon="map-marker")
                ).add_to(m)
                bounds_points.append([float(m_lat), float(m_long)])
                markers_drawn += 1

        # Process polygon_analysis
        polygon_analysis = data.get('polygon_analysis', [])
        if not polygon_analysis and '_value' in data:
            polygon_analysis = data['_value'].get('polygon_analysis', [])
        if not polygon_analysis and main_analysis:
            polygon_analysis = main_analysis.get('polygon_analysis', [])
            
        polygons_drawn = 0
        rich_polygons_drawn = 0
        rich_markers_drawn = 0
        
        for poly_item in polygon_analysis:
            pincode = poly_item.get('pincode', 'Unknown Pincode')
            boundary_polygon = poly_item.get('boundary_polygon', [])
            
            for ring in boundary_polygon:
                # Folium Polygon expects [[lat, lng], ...]
                converted_ring = [[p[1], p[0]] for p in ring]
                
                folium.Polygon(
                    locations=converted_ring,
                    color='cadetblue',
                    weight=3,
                    fill=True,
                    fill_color='lightblue',
                    fill_opacity=0.4,
                    popup=folium.Popup(f"Pincode: {pincode}", max_width=300),
                    tooltip=f"Pincode: {pincode}"
                ).add_to(m)
                bounds_points.extend(converted_ring)
                polygons_drawn += 1
                
            # Process rich aggregated_insights
            aggregated_insights = poly_item.get('aggregated_insights', {})
            for hex_id, details in aggregated_insights.items():
                insight_key = details.get('insight_key', {})
                markers = details.get('markers', [])
                
                # Render hex as polygon if h3 available
                if H3_AVAILABLE:
                    try:
                        try:
                            hex_boundary = h3.cell_to_boundary(hex_id)
                        except AttributeError:
                            hex_boundary = h3.h3_to_geo_boundary(hex_id)
                            
                        insight_content = f"<b>Hex ID:</b> {hex_id}<br>"
                        for ds, metrics in insight_key.items():
                            insight_content += f"<br><b>Dataset: {ds}</b><br>"
                            for k, v in metrics.items():
                                insight_content += f"- {k}: {v}<br>"
                                
                        folium.Polygon(
                            locations=hex_boundary,
                            color='gray',
                            weight=2,
                            fill=True,
                            fill_color='gray',
                            fill_opacity=0.3,
                            popup=folium.Popup(insight_content, max_width=400),
                            tooltip=f"Hex: {hex_id}"
                        ).add_to(m)
                        bounds_points.extend(hex_boundary)
                        rich_polygons_drawn += 1
                    except Exception as e:
                         pass
                         
                # Render markers inside the hex
                for marker in markers:
                    m_lat = marker.get('lat')
                    m_lon = marker.get('lon')
                    title = marker.get('title', 'Marker')
                    m_type = marker.get('type', 'Unknown')
                    
                    if m_lat and m_lon:
                        color = 'gray'
                        icon_name = 'info-sign'
                        if m_type == 'Competitor':
                            color = 'red'
                        elif m_type == 'TransportHub':
                            color = 'blue'
                        elif m_type == 'DemandCenter':
                            color = 'green'
                            
                        folium.Marker(
                            [float(m_lat), float(m_lon)],
                            popup=folium.Popup(f"<b>{title}</b><br>Type: {m_type}", max_width=300),
                            tooltip=title,
                            icon=folium.Icon(color=color, icon=icon_name)
                        ).add_to(m)
                        bounds_points.append([float(m_lat), float(m_lon)])
                        rich_markers_drawn += 1

        if polygons_drawn > 0:
            print(f"Drawn {polygons_drawn} boundary polygons on the map.")
        if rich_polygons_drawn > 0:
            print(f"Drawn {rich_polygons_drawn} rich hexagons on the map.")
        if rich_markers_drawn > 0:
            print(f"Drawn {rich_markers_drawn} rich markers on the map.")

        if h3_drawn > 0:
            print(f"Drawn {h3_drawn} hexagons (dataset insights/competitor insights) on the map.")
        if markers_drawn > 0:
            print(f"Drawn {markers_drawn} markers on the map.")
            
        # Fit bounds if we have points
        if bounds_points:
            m.fit_bounds(bounds_points)
            print(f"Fitted map to bounds covering {len(bounds_points)} points.")

        # 4. Display the map in the notebook
        return m

    except requests.exceptions.ConnectionError:
        print("Error: Could not connect to the API. Is your local server running on port 8000?")
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f"An error occurred: {e}")


In [ ]:
import json
import os

# Available data files: default.json, session-kandivaliborivaliandheri.json, session-thane.json
file_path = f"{formatted_time}.json"
# file_path="default.json"
# file_path = "/usr/local/google/home/vbali/Downloads/session-thane.json"

if not os.path.exists(file_path):
    file_path = "default.json"

print(f"Reading data from: {file_path}")
with open(file_path, 'r') as f:
    data_source = json.load(f)
    # Printing keys to avoid huge output
    print("Keys in JSON:", data_source.keys())
    if "state" in data_source:
        print("Keys in 'state':", data_source["state"].keys())


Reading data from: default.json
Keys in JSON: dict_keys(['_ag_ui_thread_id', '_ag_ui_app_name', '_ag_ui_user_id', 'competitor_analysis', 'polygon_analysis'])


In [ ]:
# Call the function to show the map
# map_object = display_map_from_api("http://localhost:8000/api/state")

# Passing data_source.get('state', data_source) because the file structure seems to contain "state" key
# If "state" is not present, it will fallback to the root object.
map_object = display_map_from_api(data_source.get('state', data_source))
map_object


Drawn 1 boundary polygons on the map.
Drawn 9 hexagons (dataset insights/competitor insights) on the map.
Drawn 40 markers on the map.
Fitted map to bounds covering 114 points.


In [ ]:
import os

In [ ]:
os.cpu_count()

8